In [0]:
#connect mySQL JDBC
host = "aws-0-ap-southeast-1.pooler.supabase.com"
port = 5432
database = "postgres"
jdbc_url = (
    f"jdbc:postgresql://{host}:{port}/{database}"
    "?sslmode=require"
)

connection_properties = {
    "user": "postgres.kcidyqjaxlyhhoqxcxbb",
    "password": "auth2026soa",
    "driver": "org.postgresql.Driver"
}

In [0]:
#read data
df_refresh_token = spark.read.jdbc(url=jdbc_url, table="refresh_tokens", properties=connection_properties)
display(df_refresh_token)

In [0]:
#write data
df_refresh_token.printSchema()
from pyspark.sql.functions import current_timestamp, lit

df_bronze = (
    df_refresh_token
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source", lit("supabase_postgresql"))
)

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.refresh_tokens_bronze")

In [0]:
%sql
SELECT *
FROM workspace.default.refresh_tokens_bronze;

In [0]:
%sql
SHOW TABLES IN workspace.default;

In [0]:
api_key = dbutils.secrets.get(
    scope="supabase-secrets",
    key="storage-api-key"
)

print("Đọc Secret thành công!")

In [0]:
access_key = dbutils.secrets.get(
    "supabase-secrets", "s3-access-key"
)

secret_key = dbutils.secrets.get(
    "supabase-secrets", "s3-secret-key"
)

In [0]:
%pip install boto3

In [0]:
#connect databricks to supabase storage with S3 protocol
#authencation using access key and secret key
#list the files in the bucket
import boto3
from botocore.config import Config

s3 = boto3.client(
    "s3",
    endpoint_url=(
        "https://kcidyqjaxlyhhoqxcxbb.storage.supabase.co"
        "/storage/v1/s3"
    ),
    region_name="ap-southeast-1",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path"}
    )
)

response = s3.list_objects_v2(
    Bucket="raw-data",
    MaxKeys=100
)

for obj in response.get("Contents", []):
    print(obj["Key"])

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.raw_files;

In [0]:
#download the file to databricks
import os

volume_path = "/Volumes/workspace/default/raw_files/customers"
os.makedirs(volume_path, exist_ok=True)

s3.download_file(
    "raw-data",
    "Customers.csv",
    f"{volume_path}/customers_001.csv"
)

print("Download thành công!")

In [0]:
#using Auto Load to ingest data from S3 to Bronze
#create schemas for Auto Loader remember data structure
# Auto Loader lưu schema để biết dữ liệu có những cột nào.
# Nếu sau này file mới có thêm cột phone, Auto Loader có thể phát hiện thay đổi schema và xử lý theo cấu hình schema evolution.
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/workspace/default/raw_files/schemas/customers"
    )
    .load(volume_path)
)
df_stream.printSchema()

In [0]:
#write data to bronze
#create checkpoint/customers for Auto Loader remembers ingestion progress
#Giúp Auto Loader ghi nhớ tiến độ ingestion, tránh xử lý lại những file đã được ghi thành công khi chạy lại pipeline.
from pyspark.sql.functions import current_timestamp, lit

df_bronze = (
    df_stream
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source", lit("supabase_s3"))
)

query = (
    df_bronze.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/raw_files/checkpoints/customers"
    )
    .trigger(availableNow=True)
    .toTable("workspace.default.customers_bronze")
)

query.awaitTermination()

print("Ingest vào Bronze thành công!")